In [20]:
import pandas as pd
from pprint import pprint
import json

In [5]:
df = pd.read_csv('test.csv')

In [9]:
data = df.iloc[0]

In [10]:
data

chart_id                                           10x2rgiqw97wdspi
question          which season was above average in both commerc...
answer                                    2016/17, 2017/18, 2018/19
explanation       The average revenue for Commercial is about 24...
module1_input     {"fields": ["Season", "Revenue_Type", "Revenue...
grammar           {"ops": [{"op": "average", "id": "n1", "meta":...
module1_output    {"nodes": [{"nodeId": "n1", "op": "average", "...
module2_output    {"nodes": [{"nodeId": "n1", "op": "average", "...
Name: 0, dtype: object

In [21]:
json.loads(data.module1_input)

{'fields': ['Season', 'Revenue_Type', 'Revenue_Million_Euros'],
 'dimension_fields': ['Season', 'Revenue_Type'],
 'measure_fields': ['Revenue_Million_Euros'],
 'primary_dimension': 'Season',
 'primary_measure': 'Revenue_Million_Euros',
 'series_field': 'Revenue_Type',
 'categorical_values': {'Season': ['2009/10',
   '2010/11',
   '2011/12',
   '2012/13',
   '2013/14',
   '2014/15',
   '2015/16',
   '2016/17',
   '2017/18',
   '2018/19',
   '2019/20'],
  'Revenue_Type': ['Broadcasting', 'Commercial', 'Matchday']},
 'field_types': {'Season': 'categorical',
  'Revenue_Type': 'categorical',
  'Revenue_Million_Euros': 'numeric'},
 'numeric_stats': {'Revenue_Million_Euros': {'min': 108.2,
   'max': 359.6,
   'mean': 199.11515151515152}},
 'mark': 'bar',
 'is_stacked': True,
 'encoding_summary': {'x': {'field': 'Season',
   'type': 'nominal',
   'title': None,
   'aggregate': None,
   'stack': None},
  'y': {'field': 'Revenue_Million_Euros',
   'type': 'quantitative',
   'title': None,
   'ag

### Question: Which months have above-average count in both rain and sun?
### Explanation:

Compute the average of count for rain and for sun across all months. Filter months where each is above its own average. Take the intersection of the two month sets.


# Pre-Stage: Building Context
### Input: VL Spec + CSV
### Output: ChartContext
```json
{
  "primary_dimension": "month",
  "primary_measure": "count",
  "series_field": "weather",
  "fields": ["month", "weather", "count"],
  "measure_fields": ["count"],
  "dimension_fields": ["month", "weather"],
  "categorical_values": {
    "month": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"],
    "weather": ["drizzle", "fog", "rain", "snow", "sun"]
  }
}
```

# Stage (Module) 1 — Explanation Decomposition
### Input: Chart contest, question explanation
### Output: PlanTree (temp)

```json
{
    "nodes": [
      {
        "nodeId": "n1",
        "op": "average",
        "group": "ops",
        "params": { "field": "@primary_measure", "group": "Rain" },
        "inputs": [],
        "sentenceIndex": 1
      },
      {
        "nodeId": "n2",
        "op": "average",
        "group": "ops",
        "params": { "field": "@primary_measure", "group": "Sun" },
        "inputs": [],
        "sentenceIndex": 1
      },
      {
        "nodeId": "n3",
        "op": "filter",
        "group": "ops2",
        "params": { "field": "@primary_measure", "operator": ">", "value": "ref:n1", "group": "Rain" },
        "inputs": ["n1"],
        "sentenceIndex": 2
      },
      {
        "nodeId": "n4",
        "op": "filter",
        "group": "ops2",
        "params": { "field": "@primary_measure", "operator": ">", "value": "ref:n2", "group": "Sun" },
        "inputs": ["n2"],
        "sentenceIndex": 2
      },
      {
        "nodeId": "n5",
        "op": "setOp",
        "group": "ops3",
        "params": { "fn": "intersection" },
        "inputs": ["n3", "n4"],
        "sentenceIndex": 3
      }
    ],
    "warnings": []
}
```

# Stage (Module) 2 - Chart Grounded Resolution
### Input: PlanTree, ChartContext
### Output: 
 ```json
{
  "grounded_plan_tree": {
    "nodes": [
      {
        "nodeId": "n1",
        "op": "average",
        "group": "ops",
        "params": { "field": "count", "group": "rain" },
        "inputs": [],
        "sentenceIndex": 1
      },
      {
        "nodeId": "n2",
        "op": "average",
        "group": "ops",
        "params": { "field": "count", "group": "sun" },
        "inputs": [],
        "sentenceIndex": 1
      },
      {
        "nodeId": "n3",
        "op": "filter",
        "group": "ops2",
        "params": { "field": "count", "operator": ">", "value": "ref:n1", "group": "rain" },
        "inputs": ["n1"],
        "sentenceIndex": 2
      },
      {
        "nodeId": "n4",
        "op": "filter",
        "group": "ops2",
        "params": { "field": "count", "operator": ">", "value": "ref:n2", "group": "sun" },
        "inputs": ["n2"],
        "sentenceIndex": 2
      },
      {
        "nodeId": "n5",
        "op": "setOp",
        "group": "ops3",
        "params": { "fn": "intersection" },
        "inputs": ["n3", "n4"],
        "sentenceIndex": 3
      }
    ],
    "warnings": []
  },
  "warnings": [],
  "llm_called": false
}
```

# Stage (Module) 3 - Grammar Specification
### Input: PlanTree (Grounded from Stage 2)
### Output: Speficification
```json
{
  "ops_spec": {
    "ops": [
      {
        "op": "average",
        "id": "n1",
        "meta": { "nodeId": "n1", "inputs": [], "sentenceIndex": 1 },
        "field": "count",
        "group": "rain"
      },
      {
        "op": "average",
        "id": "n2",
        "meta": { "nodeId": "n2", "inputs": [], "sentenceIndex": 1 },
        "field": "count",
        "group": "sun"
      }
    ],
    "ops2": [
      {
        "op": "filter",
        "id": "n3",
        "meta": { "nodeId": "n3", "inputs": ["n1"], "sentenceIndex": 2 },
        "field": "count",
        "operator": ">",
        "value": "ref:n1",
        "group": "rain"
      },
      {
        "op": "filter",
        "id": "n4",
        "meta": { "nodeId": "n4", "inputs": ["n2"], "sentenceIndex": 2 },
        "field": "count",
        "operator": ">",
        "value": "ref:n2",
        "group": "sun"
      }
    ],
    "ops3": [
      {
        "op": "setOp",
        "id": "n5",
        "meta": { "nodeId": "n5", "inputs": ["n3", "n4"], "sentenceIndex": 3 },
        "fn": "intersection"
      }
    ]
  },
  "warnings": []
}
```